In [0]:
# =============================================================================
# Notebook: 06_optimize_benchmark
# Purpose : Benchmark join performance before and after applying Spark
#           optimization techniques (partitioning, caching, broadcast joins).
#           Produces real, measured runtime numbers - not assumed ones.
# =============================================================================

from pyspark.sql import functions as F
import time

storage_account = "stretailcdcproj"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/retail_orders/"

# ---- Step 1: Read Silver data ----
df_silver = spark.read.format("delta").load(silver_path)
print(f"Rows in silver: {df_silver.count()}")

# ---- Step 2: Scale up the dataset synthetically ----
# 5,100 rows is too small to show a meaningful timing difference between
# optimized vs unoptimized joins. We simulate a larger production-like volume
# by duplicating the data with new unique order_ids - this is ONLY for
# benchmarking purposes, not part of the real CDC pipeline.
df_large = df_silver
for i in range(6):  # doubles data ~6 times -> ~5100 * 64 ≈ 326,000 rows
    df_large = df_large.union(
        df_large.withColumn("order_id", F.col("order_id") + F.lit(1000000 * (i + 1)))
    )

large_count = df_large.count()
print(f"Simulated large dataset size: {large_count} rows")

# ---- Step 3: Create a small lookup/dimension table ----
# Simulates a product master table - small enough to be a broadcast candidate.
products = ['Shampoo', 'Conditioner', 'Face Cream', 'Lipstick', 'Serum', 'Sunscreen']
categories = ['Hair Care', 'Hair Care', 'Skin Care', 'Makeup', 'Skin Care', 'Skin Care']
suppliers = ['SupplierA', 'SupplierB', 'SupplierA', 'SupplierC', 'SupplierB', 'SupplierC']

df_lookup = spark.createDataFrame(
    list(zip(products, categories, suppliers)),
    ["product", "category", "supplier"]
)
print(f"Lookup table size: {df_lookup.count()} rows")
df_lookup.show()

# ---- Step 4: no optimizations ----
# Spark uses its default strategy (typically a shuffle/sort-merge join for this join type).
start_time = time.time()

df_joined_baseline = df_large.join(df_lookup, on="product", how="inner")
baseline_result_count = df_joined_baseline.count()  # action to force execution

baseline_time = time.time() - start_time

print(f"\n--- BASELINE (no optimization) ---")
print(f"Joined row count: {baseline_result_count}")
print(f"Runtime: {baseline_time:.2f} seconds")

Rows in silver: 5100
Simulated large dataset size: 326400 rows
Lookup table size: 6 rows
+-----------+---------+---------+
|    product| category| supplier|
+-----------+---------+---------+
|    Shampoo|Hair Care|SupplierA|
|Conditioner|Hair Care|SupplierB|
| Face Cream|Skin Care|SupplierA|
|   Lipstick|   Makeup|SupplierC|
|      Serum|Skin Care|SupplierB|
|  Sunscreen|Skin Care|SupplierC|
+-----------+---------+---------+


--- BASELINE (no optimization) ---
Joined row count: 326400
Runtime: 8.31 seconds


In [0]:
# ---- OPTIMIZATION 1: Broadcast Join ----
# NOTE: Databricks Serverless runs on Spark Connect architecture, which does
# NOT support the older functional API `F.broadcast()`. Instead, we use the
# DataFrame-level `.hint("broadcast")` method, which IS supported and
# achieves the same result — instructing Spark's optimizer to broadcast the
# small table to every executor instead of shuffling the large one.
start_time = time.time()

df_joined_broadcast = df_large.join(df_lookup.hint("broadcast"), on="product", how="inner")
broadcast_result_count = df_joined_broadcast.count()  # force execution

broadcast_time = time.time() - start_time

print(f"\n--- OPTIMIZATION 1: Broadcast Join ---")
print(f"Joined row count: {broadcast_result_count}")
print(f"Runtime: {broadcast_time:.2f} seconds")

# ---- Compare to baseline ----
improvement_pct = ((baseline_time - broadcast_time) / baseline_time) * 100
print(f"\n Improvement over baseline: {improvement_pct:.2f}%")
print(f"   Baseline : {baseline_time:.2f}s")
print(f"   Broadcast: {broadcast_time:.2f}s")


--- OPTIMIZATION 1: Broadcast Join ---
Joined row count: 326400
Runtime: 6.48 seconds

📊 Improvement over baseline: 21.97%
   Baseline : 8.31s
   Broadcast: 6.48s


In [0]:
# ---- OPTIMIZATION 2: Partitioning + Broadcast Join Combined ----
# Repartitioning df_large by the join key ("product") before the join can
# reduce shuffle overhead, especially when followed by additional
# transformations on the same key (e.g., aggregations downstream).
# We combine this with the broadcast hint from Optimization 1.
start_time = time.time()

df_large_repartitioned = df_large.repartition("product")

df_joined_partitioned = df_large_repartitioned.join(
    df_lookup.hint("broadcast"), on="product", how="inner"
)
partitioned_result_count = df_joined_partitioned.count()  # force execution

partitioned_time = time.time() - start_time

print(f"\n--- OPTIMIZATION 2: Partitioning + Broadcast ---")
print(f"Joined row count: {partitioned_result_count}")
print(f"Runtime: {partitioned_time:.2f} seconds")

improvement_pct_2 = ((baseline_time - partitioned_time) / baseline_time) * 100
print(f"\n Improvement over baseline: {improvement_pct_2:.2f}%")
print(f"   Baseline             : {baseline_time:.2f}s")
print(f"   Broadcast only       : {broadcast_time:.2f}s")
print(f"   Partitioning+Broadcast: {partitioned_time:.2f}s")


--- OPTIMIZATION 2: Partitioning + Broadcast ---
Joined row count: 326400
Runtime: 5.57 seconds

📊 Improvement over baseline: 33.00%
   Baseline             : 8.31s
   Broadcast only       : 6.48s
   Partitioning+Broadcast: 5.57s
